# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bajwaycodes/Kashif-Working-Repo/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
import duckdb
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"

In [21]:
con.sql(f"""
    SELECT COUNT(*) AS row_count
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
""")

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│   9841378 │
└───────────┘

In [22]:
import pandas as pd
pd.set_option("display.max_rows", None)
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [23]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — never the sealed final month (2026-06)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item, for one client, on one single
calendar day. Grain = `report_date + client_hash_id + content_hash_id`, from
`fact_content_daily_performance`.

**Time window:** March 2026 only (`month=2026-03`) — a mid-panel month, not the
`_sample` table, which is exactly June 2026 (the sealed final month) and must never be
used to build label logic.

Within March, I split further so the whole exercise stays inside this one partition:
- **Days 1–15** = my "prior" feature window (what's knowable at a decision point).
- **Days 16–31** = my "future" window, used only to build a proxy label.

This mirrors a real past→future split without needing any data outside March.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Field | Bucket | Why |
|---|---|---|
| `client_hash_id`, `content_hash_id` | Context | Join/group/split keys only — pseudonyms, never model inputs |
| `report_date`, `month` | Context | Used to build the prior/future windows, not a feature itself |
| `gsc_data_available`, `ga4_data_available` | Context | Availability flags — used to filter rows, not learned from |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` (days 1–15) | Feature | Logged daily as events happen; known by day 15, before the label window starts |
| `ga4_sessions`, `ga4_engaged_sessions` (days 1–15) | Feature | Same — daily GA4 activity, known before day 16 |
| `scroll_events` (days 1–15) | Feature | Same — daily engagement signal, known before day 16 |
| `gsc_clicks` (days 16–31) | Label source | Used ONLY to build the proxy label below — never a feature |
| `client_has_gsc`, `client_has_ga4` | Excluded | Client-level static flags, not needed for a page-day model; would just duplicate `*_data_available` |
| `sessions_paid`, `sessions_social`, `sessions_referral`, `sessions_direct` | Excluded | Out of scope for this lane's core signal (organic search decline); channel breakdown isn't part of the refresh-priority question |
| `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | Excluded | Per the flyrank-data skill, AI-session rows are extremely sparse (30,177 of 78.8M in the full table) — not reliable for this lane, would need dedicated sparse-data handling |
| `ga4_total_engagement_sec` | Excluded (for now) | Redundant with `ga4_engaged_sessions` for a 5-feature limit; noted as a candidate for later weeks |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**Query 1 — grain check.** If the grain claim is true, no group of
(date, client, content) should appear more than once.

In [24]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""")
grain_check  # expect: empty result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────┬────────────────┬─────────────────┬───────┐
│ report_date │ client_hash_id │ content_hash_id │   c   │
│    date     │    varchar     │     varchar     │ int64 │
├─────────────┴────────────────┴─────────────────┴───────┤
│                         0 rows                         │
└────────────────────────────────────────────────────────┘

**Query 2 — row count and date span** for the March 2026 partition.

In [25]:
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")
span_check

┌───────────┬────────────┬────────────┐
│ row_count │  min_date  │  max_date  │
│   int64   │    date    │    date    │
├───────────┼────────────┼────────────┤
│   9841378 │ 2026-03-01 │ 2026-03-31 │
└───────────┴────────────┴────────────┘

**Query 3 — availability.** Filtering with `IS TRUE` (not just truthy) on
`gsc_data_available`, and showing how many rows survive versus the unfiltered total.

In [26]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬────────────────────┐
│ total_rows │ gsc_available_rows │ ga4_available_rows │
│   int64    │       int64        │       int64        │
├────────────┼────────────────────┼────────────────────┤
│    9841378 │            3611061 │             413966 │
└────────────┴────────────────────┴────────────────────┘

**Five features**, built only from days 1–15 of March (the "prior" window) so every
one is knowable before the label window even starts.

In [27]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_first_half,
        SUM(gsc_clicks) AS clicks_first_half,
        AVG(gsc_avg_position) AS avg_position_first_half,
        SUM(ga4_sessions) AS sessions_first_half,
        SUM(ga4_engaged_sessions) AS engaged_sessions_first_half
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE report_date <= DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,sessions_first_half,engaged_sessions_first_half
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,NaN,NaN
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,NaN,NaN
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,NaN,NaN
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,NaN,NaN
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,240.0,1.0,3.860842,NaN,NaN


- `impressions_first_half` — knowable at the decision moment because it's a running
  sum of daily GSC impressions logged as they happen, entirely within days 1–15.
- `clicks_first_half` — same: daily click events, all before day 16.
- `avg_position_first_half` — daily position, averaged only over the days already
  observed by the decision point.
- `sessions_first_half` — daily GA4 session counts, same window.
- `engaged_sessions_first_half` — daily engaged-session counts, same window; combined
  with sessions, this gives an engagement rate that's fully known by day 15.

**The trap.** I'll build a proxy label from the *future* window (days 16–31): did a
page's clicks decline in the second half of the month versus the first half? Then
I'll deliberately add one column that's derived from the same data the label is
computed from, watch a quick score jump toward perfect, and remove it.

In [28]:
# Build the label from the future window (days 16-31) — separate from features above
label_source = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clicks_second_half
    FROM read_parquet('{rel}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE report_date > DATE '2026-03-15'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

data = features.merge(label_source, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining"] = (data["clicks_second_half"] < data["clicks_first_half"]).astype(int)
print(data["is_declining"].value_counts(normalize=True))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining
0    0.797345
1    0.202655
Name: proportion, dtype: float64


In [29]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = ["impressions_first_half", "clicks_first_half",
                    "avg_position_first_half", "sessions_first_half",
                    "engaged_sessions_first_half"]

X_honest = data[honest_features].fillna(0)
y = data["is_declining"]

model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
auc_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"Honest AUC (features only from days 1-15): {auc_honest:.3f}")

Honest AUC (features only from days 1-15): 0.746


In [30]:
# THE TRAP: add clicks_second_half itself — the exact column the label is derived from
X_leaky = data[honest_features + ["clicks_second_half"]].fillna(0)

model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
auc_leaky = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky AUC (with clicks_second_half added): {auc_leaky:.3f}  <- jumps toward perfect")

Leaky AUC (with clicks_second_half added): 1.000  <- jumps toward perfect


In [31]:
# Remove it. Keep only the honest number.
print(f"Honest AUC: {auc_honest:.3f}")
print(f"Leaky AUC:  {auc_leaky:.3f}  -- discarded, this number is fake")
print("\nKeeping the honest AUC as the real result for this slice.")

Honest AUC: 0.746
Leaky AUC:  1.000  -- discarded, this number is fake

Keeping the honest AUC as the real result for this slice.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me:
- **Anything outside March 2026** — one month of one partition, not the full
  17-month panel; seasonality or longer trends are invisible here.
- **True content-level context** — `fact_content_daily_performance` has no word
  count, content type, or age; those live in `dim_content`, which I didn't join here,
  so I can't yet connect behavioral decline to content characteristics.
- **Whether a page's history even covers all of March** — per the flyrank-data
  skill, client history depth varies wildly, and rows before a client's
  `ga4_data_start` are zero-filled with `ga4_data_available = FALSE`. I filtered on
  the flag, but I haven't checked how many client-content pairs got dropped entirely
  by that filter, versus just having some blank days.
- **Causality** — a clicks decline observed here says nothing about *why* it
  declined (SERP change, seasonality, consolidation) — that needs the checks from
  the lane guide's Section 7, which I haven't run yet.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [ ] Every section filled — markdown thinking AND the code that backs it
- [ ] Notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] Claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to `work/notebooks/w03_data_contract.ipynb` — repo URL submitted on the card